In [ ]:
import os
import re
import numpy as np
import pandas as pd
from typing import Optional
import json 
import pickle 

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix


# ══════════════════════════════════════════════════════════════════════════════
# MANUAL FINANCIALS DICTIONARY
# Add every movie you have CSVs for here.
# Key must match the movie name portion of your filename exactly
# (case-insensitive, underscores or spaces both work).
#
# Format:  "movie_name": {"budget": <int>, "gross": <int>}
#
# Later, once the scraper can pull budget/gross, you only need to replace the
# get_financials() function below — nothing else in the codebase changes.
# ══════════════════════════════════════════════════════════════════════════════

MOVIE_FINANCIALS = {
    "moonlight":       {"budget": 1_500_000,   "gross": 27_854_932},
    "parasite":        {"budget": 11_400_000,  "gross": 53_369_749},
    "mulholland_drive":{"budget": 15_000_000,  "gross": 7_220_243},
}


def get_financials(movie_name: str) -> dict:
    """
    Look up budget and gross for a movie by name.

    For now: reads from the MOVIE_FINANCIALS dict above.
    Later: replace this function body with a CSV/DB lookup —
    everything that calls it stays exactly the same.
    """
    # normalize: lowercase, replace spaces with underscores for consistent matching
    key = movie_name.lower().strip().replace(" ", "_")
    return MOVIE_FINANCIALS.get(key, {"budget": np.nan, "gross": np.nan})


# ══════════════════════════════════════════════════════════════════════════════
# CSV LOADING
# Filename format expected: reviews_[platform]_moviename_trial[N].csv
# Examples:
#   reviews_imdb_oppenheimer_trial1.csv
#   reviews_RT_the_flash_trial2.csv
#   reviews_letter_barbie_trial1.csv
# ══════════════════════════════════════════════════════════════════════════════

# known platform tags to strip when parsing movie name from filename
KNOWN_PLATFORMS = {"imdb", "rt", "letter", "letterboxd", "metacritic", "rottentomatoes"}


def parse_movie_name(filename: str) -> str:
    name  = os.path.splitext(filename)[0].lower()   # drop .csv
    # strip known platform tags anywhere in the name
    for platform in KNOWN_PLATFORMS:
        name = re.sub(rf"_?{platform}_?", "_", name)
    # strip trailing '_reviews' suffix
    name = re.sub(r"_?reviews_?$", "", name)
    # clean up any double underscores left behind
    name = re.sub(r"_+", "_", name).strip("_")
    return name


def parse_user_rating(val) -> float:
    """
    Robustly convert user_rating to a float regardless of format.
    Handles: 8.5, '8.5', '8.5/10', '85%', '4/5', 'N/A', None, NaN
    All unresolvable values return NaN.
    """
    if pd.isna(val):
        return np.nan
    s = str(val).strip()

    # percentage: '85%' -> normalize to /10 scale
    pct = re.match(r"^([\d.]+)%$", s)
    if pct:
        return float(pct.group(1)) / 10.0

    # fraction: '8.5/10' or '4/5' -> normalize to /10 scale
    frac = re.match(r"^([\d.]+)\s*/\s*([\d.]+)$", s)
    if frac:
        num, denom = float(frac.group(1)), float(frac.group(2))
        if denom == 0:
            return np.nan
        return (num / denom) * 10.0

    # plain number: '8.5', '7'
    try:
        return float(s)
    except ValueError:
        return np.nan


def load_all_csvs(folder_path: str) -> pd.DataFrame:
    """
    Load all CSVs from a folder, parse the movie name from each filename,
    attach budget/gross from MOVIE_FINANCIALS, and return one combined DataFrame.

    Expected columns per CSV: id, author, date, user_rating, review_title, review_text
    Added columns:            movie_title, platform, budget, gross
    """
    all_dfs   = []
    csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in: {folder_path}")

    for fname in csv_files:
        fpath = os.path.join(folder_path, fname)
        try:
            df = pd.read_csv(fpath)
        except Exception as e:
            print(f"  [skip] could not read {fname}: {e}")
            continue

        # parse movie name from filename
        movie_name = parse_movie_name(fname)

        # extract platform tag from filename
        raw      = re.sub(r"^reviews_", "", os.path.splitext(fname)[0].lower())
        platform = raw.split("_")[0] if raw.split("_")[0] in KNOWN_PLATFORMS else "unknown"

        df["movie_title"] = movie_name
        df["platform"]    = platform

        # attach financials from the manual dict above
        financials   = get_financials(movie_name)
        df["budget"] = financials["budget"]
        df["gross"]  = financials["gross"]

        # normalize user_rating to a consistent float on /10 scale
        if "user_rating" in df.columns:
            df["user_rating"] = df["user_rating"].apply(parse_user_rating)
        else:
            df["user_rating"] = np.nan

        all_dfs.append(df)
        print(f"  [loaded] {fname}  ->  movie: '{movie_name}'  |  rows: {len(df)}")

    if not all_dfs:
        raise ValueError("All CSV files failed to load.")

    combined = pd.concat(all_dfs, ignore_index=True)
    print(f"\nTotal rows loaded:   {len(combined)}")
    print(f"Unique movies found: {combined['movie_title'].nunique()}")
    return combined


# ══════════════════════════════════════════════════════════════════════════════
# LABEL ASSIGNMENT
# Hit = gross >= 3x budget.  Flop = everything else.
# Rows where financials are missing are dropped.
# ══════════════════════════════════════════════════════════════════════════════

def assign_label(row: pd.Series) -> float:
    budget = row.get("budget", np.nan)
    gross  = row.get("gross",  np.nan)
    if pd.notna(budget) and budget > 0 and pd.notna(gross):
        return 1 if (gross / budget) >= 3.0 else 0
    return np.nan


def label_dataset(df: pd.DataFrame) -> pd.DataFrame:
    df        = df.copy()
    df["label"] = df.apply(assign_label, axis=1)
    before    = len(df)
    df        = df.dropna(subset=["label"])
    dropped   = before - len(df)
    if dropped:
        print(f"  [labels] dropped {dropped} movies with missing financials "
              f"(add them to MOVIE_FINANCIALS to include them)")
    df["label"] = df["label"].astype(int)
    return df


# ══════════════════════════════════════════════════════════════════════════════
# PREPROCESSING
# Collapses to one row per movie.
# Numeric features: avg_user_rating for now.
# To add more later (critic score, ROI etc.) just append to NUMERIC_FEATURES —
# the model reads num_dim dynamically so nothing else needs to change.
# ══════════════════════════════════════════════════════════════════════════════

NUMERIC_FEATURES = [
    "avg_user_rating",
    # future additions — uncomment when scraper can provide them:
    # "avg_critic_score",
    # "roi",
]


def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    df  = df.copy()

    agg = {
        # join all reviews with [SEP] so BERT can distinguish between them
        "review_text": lambda x: " [SEP] ".join(x.dropna().astype(str)),
        # average the numeric score across all reviews for this movie
        "user_rating": "mean",
        # budget/gross are the same for every row of a movie
        "budget":      "first",
        "gross":       "first",
    }

    # only aggregate columns that actually exist in this dataframe
    agg      = {k: v for k, v in agg.items() if k in df.columns}
    movie_df = df.groupby("movie_title").agg(agg).reset_index()

    # rename so downstream code is explicit about what this column contains
    if "user_rating" in movie_df.columns:
        movie_df = movie_df.rename(columns={"user_rating": "avg_user_rating"})

    movie_df = label_dataset(movie_df)
    return movie_df


def build_numeric_matrix(movie_df: pd.DataFrame, scaler: Optional[StandardScaler] = None):
    """Return scaled numeric array + fitted scaler."""
    cols  = [c for c in NUMERIC_FEATURES if c in movie_df.columns]
    X_num = movie_df[cols].fillna(0).values.astype(np.float32)
    if scaler is None:
        scaler = StandardScaler()
        X_num  = scaler.fit_transform(X_num)
    else:
        X_num  = scaler.transform(X_num)
    return X_num, scaler, cols


# ══════════════════════════════════════════════════════════════════════════════
# DATASET
# ══════════════════════════════════════════════════════════════════════════════

class MovieDataset(Dataset):
    def __init__(self, texts, numeric_features, labels, tokenizer, max_len=256):
        self.texts   = texts
        self.numeric = torch.tensor(numeric_features, dtype=torch.float32)
        self.labels  = torch.tensor(labels,           dtype=torch.float32)
        self.tok     = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx],
            max_length=self.max_len,
            truncation=True,        # drop tokens beyond max_len
            padding="max_length",   # pad shorter sequences so all batches are same shape
            return_tensors="pt",
        )
        return {
            # squeeze(0) removes the extra batch dimension the tokenizer adds
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "numeric":        self.numeric[idx],
            "label":          self.labels[idx],
        }


# ══════════════════════════════════════════════════════════════════════════════
# MODEL
# ══════════════════════════════════════════════════════════════════════════════

class MovieClassifier(nn.Module):
    def __init__(self, num_dim: int, hidden_dim: int = 256, dropout: float = 0.3):
        super().__init__()

        self.bert = BertModel.from_pretrained("bert-base-uncased")

        # freeze all BERT parameters, selectively unfreeze top 2 encoder blocks.
        # lower layers = general language features (keep frozen).
        # upper layers (10, 11) = task-specific (fine-tune these).
        for name, param in self.bert.named_parameters():
            if not any(f"encoder.layer.{i}" in name for i in [10, 11]):
                param.requires_grad = False

        bert_dim = self.bert.config.hidden_size   # 768 for bert-base-uncased

        # numeric MLP — compresses features to 64-d
        self.numeric_branch = nn.Sequential(
            nn.Linear(num_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
        )

        # fusion MLP — takes 768 + 64 = 832-d input, outputs 1 logit
        fusion_in = bert_dim + 64
        self.fusion = nn.Sequential(
            nn.Linear(fusion_in, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),   # raw logit, no activation
        )

    def forward(self, input_ids, attention_mask, numeric):
        bert_out      = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        text_features = bert_out.last_hidden_state[:, 0, :]            # [B, 768]
        num_features  = self.numeric_branch(numeric)                   # [B, 64]
        combined      = torch.cat([text_features, num_features], dim=1)  # [B, 832]
        output        = self.fusion(combined)                          # [B, 1]
        return output.squeeze(1)                                       # [B]


# ══════════════════════════════════════════════════════════════════════════════
# TRAINING & EVALUATION
# ══════════════════════════════════════════════════════════════════════════════

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attn_mask = batch["attention_mask"].to(device)
        numeric   = batch["numeric"].to(device)
        labels    = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attn_mask, numeric)
        loss   = criterion(logits, labels)
        loss.backward()

        # clip gradients — prevents exploding gradients during BERT fine-tuning
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        preds       = (torch.sigmoid(logits) >= 0.5).float()
        total_loss += loss.item() * labels.size(0)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attn_mask = batch["attention_mask"].to(device)
        numeric   = batch["numeric"].to(device)
        labels    = batch["label"].to(device)

        logits = model(input_ids, attn_mask, numeric)
        loss   = criterion(logits, labels)

        preds       = (torch.sigmoid(logits) >= 0.5).float()
        total_loss += loss.item() * labels.size(0)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)
        all_preds.extend(preds.cpu().int().tolist())
        all_labels.extend(labels.cpu().int().tolist())

    return total_loss / total, correct / total, all_preds, all_labels


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main(
    folder_path: str,
    epochs: int     = 5,
    batch_size: int = 16,
    lr: float       = 2e-5,
    max_len: int    = 256,
    dropout: float  = 0.3,
    seed: int       = 42,
):
    torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}\n")

    # load all CSVs from folder
    print(f"Loading CSVs from: {folder_path}")
    raw_df   = load_all_csvs(folder_path)
    movie_df = preprocess(raw_df)

    print(f"\nMovies after preprocessing: {len(movie_df)}")
    print("\nLabel distribution:")
    print(movie_df["label"].value_counts().rename({0: "Flop", 1: "Hit"}))

    if len(movie_df) < 10:
        print("\n[warning] fewer than 10 labeled movies — add more CSVs and financials "
              "for better training.")

    # stratified splits preserve class balance across train / val / test
    # replace the split block in main() with this
    train_df, test_df = train_test_split(movie_df, test_size=0.34, random_state=seed)
    # skip val split entirely for now — too few samples
    val_df = test_df   # just reuse test as val so the rest of the code doesn't break

    # fit scaler on train only — prevents data leakage
    X_train, scaler, num_cols = build_numeric_matrix(train_df)
    X_val,   _,      _        = build_numeric_matrix(val_df,  scaler)
    X_test,  _,      _        = build_numeric_matrix(test_df, scaler)
    print(f"\nNumeric features used: {num_cols}")

    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

    train_ds = MovieDataset(train_df["review_text"].tolist(), X_train,
                            train_df["label"].tolist(), tokenizer, max_len)
    val_ds   = MovieDataset(val_df["review_text"].tolist(),   X_val,
                            val_df["label"].tolist(),   tokenizer, max_len)
    test_ds  = MovieDataset(test_df["review_text"].tolist(),  X_test,
                            test_df["label"].tolist(),  tokenizer, max_len)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size)

    # num_dim is dynamic — auto-adjusts as you add features to NUMERIC_FEATURES
    model     = MovieClassifier(num_dim=X_train.shape[1], dropout=dropout).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_acc, best_state = 0.0, None
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc       = train_epoch(model, train_loader, optimizer, criterion, device)
        vl_loss, vl_acc, _, _ = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        print(f"Epoch {epoch}/{epochs}  "
              f"train_loss={tr_loss:.4f} train_acc={tr_acc:.3f}  "
              f"val_loss={vl_loss:.4f} val_acc={vl_acc:.3f}")

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    # restore best checkpoint, evaluate on held-out test set
    model.load_state_dict(best_state)
    model.to(device)
    _, test_acc, preds, labels = evaluate(model, test_loader, criterion, device)

    print(f"\nTest Accuracy: {test_acc:.3f}")
    print("\nClassification Report:")
    print(classification_report(labels, preds, target_names=["Flop", "Hit"]))
    print("Confusion Matrix:")
    print(confusion_matrix(labels, preds))

    # save model weights as a regular file
    torch.save(best_state, "model_weights.pt")  # weights still need .pt, that's unavoidable

    # save scaler and num_cols as a pickle file (standard Python format)
    with open("scaler.pkl", "wb") as f:
        pickle.dump({"scaler": scaler, "num_cols": num_cols}, f)

    # save num_cols as a plain JSON so you can read it like a normal text file
    with open("num_cols.json", "w") as f:
        json.dump(num_cols, f)

    print("Saved: model_weights.pt, scaler.pkl, num_cols.json")


# ══════════════════════════════════════════════════════════════════════════════
# INFERENCE
# ══════════════════════════════════════════════════════════════════════════════

def predict_movie(
    model: MovieClassifier,
    scaler: StandardScaler,
    num_cols: list,
    review_texts: list[str],
    numeric_data: dict,
    tokenizer=None,
    device=None,
    max_len: int = 256,
) -> dict:
    """
    Predict a single movie at inference time.

    Parameters
    ----------
    review_texts : list of review strings for this movie
    numeric_data : dict with keys matching num_cols, e.g. {"avg_user_rating": 7.4}
                   missing keys default to 0
    """
    if tokenizer is None:
        tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    combined_text = " [SEP] ".join(review_texts)
    num_vec       = np.array([[numeric_data.get(c, 0.0) for c in num_cols]], dtype=np.float32)
    num_vec       = scaler.transform(num_vec)   # use training scaler, never refit

    enc = tokenizer(combined_text, max_length=max_len, truncation=True,
                    padding="max_length", return_tensors="pt")
    model.eval()
    with torch.no_grad():
        logit = model(
            enc["input_ids"].to(device),
            enc["attention_mask"].to(device),
            torch.tensor(num_vec, dtype=torch.float32).to(device),
        )

    prob_hit = torch.sigmoid(logit).item()
    return {
        "prediction":    "Hit" if prob_hit >= 0.5 else "Flop",
        "probabilities": {
            "Hit":  prob_hit,
            "Flop": 1 - prob_hit,
        },
    }


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# Usage: python movie_classifier.py /path/to/csv/folder
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    import sys
    folder = sys.argv[1] if len(sys.argv) > 1 else "./data"
    main(('/Users/Diane/Desktop/PSYCH 186B/project/reviews'))

Using device: cpu

Loading CSVs from: /Users/Diane/Desktop/PSYCH 186B/project/reviews
  [loaded] moonlight_imdb_reviews.csv  ->  movie: 'moonlight'  |  rows: 50
  [loaded] parasite_imdb_reviews.csv  ->  movie: 'parasite'  |  rows: 50
  [loaded] mulholland_drive_imdb_reviews.csv  ->  movie: 'mulholland_drive'  |  rows: 50

Total rows loaded:   150
Unique movies found: 3

Movies after preprocessing: 3

Label distribution:
label
Hit     2
Flop    1
Name: count, dtype: int64

[warning] fewer than 10 labeled movies — add more CSVs and financials for better training.

Numeric features used: ['avg_user_rating']


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Epoch 1/5  train_loss=0.7267 train_acc=0.000  val_loss=0.6893 val_acc=0.500
Epoch 2/5  train_loss=0.6914 train_acc=1.000  val_loss=0.6885 val_acc=0.500
Epoch 3/5  train_loss=0.7026 train_acc=0.000  val_loss=0.6873 val_acc=0.500
Epoch 4/5  train_loss=0.7759 train_acc=0.000  val_loss=0.6867 val_acc=0.500
Epoch 5/5  train_loss=0.7241 train_acc=0.000  val_loss=0.6867 val_acc=0.500

Test Accuracy: 0.500

Classification Report:
              precision    recall  f1-score   support

        Flop       0.50      1.00      0.67         1
         Hit       0.00      0.00      0.00         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2

Confusion Matrix:
[[1 0]
 [1 0]]


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Model saved to movie_classifier.pt
